# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import duckdb
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN '')")  # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

**Ranking, via a scoring model — with a rollup step first.** The raw warehouse table is
page-day grain (one row per page per day), not page grain. To rank "which pages need review,"
I first have to aggregate a trailing window (e.g. last 90 days) per (client_hash_id,
content_hash_id) into page-level features, then score and sort those. The decision itself is
still relative order, not a per-page yes/no — same reasoning as before, just built on raw
daily rows instead of a pre-aggregated CSV.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(con.sql(f"""
    SELECT COUNT(DISTINCT content_hash_id) AS unique_pages,
           COUNT(DISTINCT client_hash_id) AS unique_clients,
           MIN(report_date) AS earliest_date,
           MAX(report_date) AS latest_date
    FROM read_parquet('{table}')
"""))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┬────────────────┬───────────────┬─────────────┐
│ unique_pages │ unique_clients │ earliest_date │ latest_date │
│    int64     │     int64      │     date      │    date     │
├──────────────┼────────────────┼───────────────┼─────────────┤
│       427292 │             70 │ 2025-01-27    │ 2026-06-30  │
└──────────────┴────────────────┴───────────────┴─────────────┘



**Still a proxy — but now a real two-window comparison instead of a same-window label.**
The starter CSV's `trend_direction` was computed from `trend_pct` in the same window as the
features, which is why w01 flagged it as circular. Here, with real `report_date` values, the
target can be built honestly: compare `gsc_clicks`/`gsc_impressions` summed over a recent
window (e.g. last 30 days) against a prior, non-overlapping window (the 30 days before that)
for the same page. "Declining" = recent window meaningfully below the prior window. It's
still a proxy for "worth a reviewer's time," not a guaranteed future outcome — but it's no
longer derived from the same data used to predict it.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(con.sql(f"""
    SELECT content_hash_id,
           SUM(CASE WHEN report_date >= (SELECT MAX(report_date) FROM read_parquet('{table}')) - INTERVAL 30 DAY
                     THEN gsc_clicks END) AS clicks_recent_30d,
           SUM(CASE WHEN report_date < (SELECT MAX(report_date) FROM read_parquet('{table}')) - INTERVAL 30 DAY
                     AND report_date >= (SELECT MAX(report_date) FROM read_parquet('{table}')) - INTERVAL 60 DAY
                     THEN gsc_clicks END) AS clicks_prior_30d
    FROM read_parquet('{table}')
    GROUP BY content_hash_id
    LIMIT 10
"""))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┬───────────────────┬──────────────────┐
│     content_hash_id      │ clicks_recent_30d │ clicks_prior_30d │
│         varchar          │      int128       │      int128      │
├──────────────────────────┼───────────────────┼──────────────────┤
│ content_fe8e8155ce1d47a2 │                 0 │                0 │
│ content_b4462a1b90640058 │                 0 │                0 │
│ content_c782fa8abd4fce5e │                 0 │                0 │
│ content_89c6c2e17e412e20 │                 0 │                0 │
│ content_8a3ccd5a0b61b7f7 │                 0 │                0 │
│ content_d39ef5d4aa3d73de │                 0 │                0 │
│ content_73d0d66253d25ab9 │                 0 │                0 │
│ content_eebfb2913c3a0eaf │                 0 │                0 │
│ content_63c7a1498f76797f │                 0 │                0 │
│ content_519c6ac7943e296b │                 0 │                0 │
├──────────────────────────┴───────────────────┴

**Precision@50, unchanged in principle** — reviewer capacity is still the real constraint,
so ranking quality at the top matters more than accuracy across every page. What changes is
what it's measured against: instead of the starter CSV's baked-in label, it now has to be
computed against the two-window proxy from Section 2, and "good" means beating whatever a
simple single-window rule achieves once evaluated with a proper client-holdout split — the
same standard established on the starter data, just re-proven here since real data can behave
differently (sparser clicks, more zero-click page-days, real seasonality by `report_date`).

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Success metric: Precision@50 against the two-window decline proxy, evaluated on held-out clients.")
print("Numeric baseline TBD once the label query in Section 2 is confirmed against real dates.")


Success metric: Precision@50 against the two-window decline proxy, evaluated on held-out clients.
Numeric baseline TBD once the label query in Section 2 is confirmed against real dates.


**Raw table: one row = one page, for one client, on one day** (`content_hash_id` +
`client_hash_id` + `report_date`). **Analysis unit for the ranking task: one page, for one
client** — built by aggregating the daily rows over a trailing window. These are different
grains, and that distinction matters: querying the raw table without grouping would silently
treat every page-day as its own "case," badly overcounting pages with more history.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(con.sql(f"""
    SELECT client_hash_id, content_hash_id, COUNT(*) AS n_days,
           MIN(report_date) AS first_day, MAX(report_date) AS last_day
    FROM read_parquet('{table}')
    GROUP BY client_hash_id, content_hash_id
    ORDER BY n_days DESC
    LIMIT 5
"""))



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬────────┬────────────┬────────────┐
│     client_hash_id      │     content_hash_id      │ n_days │ first_day  │  last_day  │
│         varchar         │         varchar          │ int64  │    date    │    date    │
├─────────────────────────┼──────────────────────────┼────────┼────────────┼────────────┤
│ client_9958f0a7ae1df715 │ content_48b01ea98ec1ea03 │    520 │ 2025-01-27 │ 2026-06-30 │
│ client_9958f0a7ae1df715 │ content_3d9b660fe6946d8a │    520 │ 2025-01-27 │ 2026-06-30 │
│ client_9958f0a7ae1df715 │ content_6c845eb06ed5c8a8 │    520 │ 2025-01-27 │ 2026-06-30 │
│ client_9958f0a7ae1df715 │ content_61cb637b1e493b91 │    520 │ 2025-01-27 │ 2026-06-30 │
│ client_9958f0a7ae1df715 │ content_3820ce7ae34747f8 │    520 │ 2025-01-27 │ 2026-06-30 │
└─────────────────────────┴──────────────────────────┴────────┴────────────┴────────────┘



**Even more true here than on the starter CSV.** The starter data already showed declining
vs. stable pages have overlapping distributions on every single signal checked (avg_position,
impressions, content age) — no clean threshold separated them. The real warehouse adds more
dimensions on top: channel-split sessions (organic/direct/referral/social/paid/AI), per-AI-
platform referral counts, and engagement metrics, all varying day to day. A rule would need
to hand-tune thresholds across dozens of correlated, trending, partially-missing signals
(note `ga4_data_available` is `false` in some sample rows — coverage itself is inconsistent
per client) simultaneously. That's exactly the kind of multi-signal interaction a model can
weigh and an if-statement can't.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(con.sql(f"""
    SELECT client_hash_id,
           AVG(CAST(gsc_data_available AS INT)) AS gsc_coverage_rate,
           AVG(CAST(ga4_data_available AS INT)) AS ga4_coverage_rate
    FROM read_parquet('{table}')
    GROUP BY client_hash_id
    ORDER BY client_hash_id
    LIMIT 10
"""))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────┬──────────────────────┐
│     client_hash_id      │  gsc_coverage_rate   │  ga4_coverage_rate   │
│         varchar         │        double        │        double        │
├─────────────────────────┼──────────────────────┼──────────────────────┤
│ client_04660893ae39614a │                  0.0 │ 0.042454008157829015 │
│ client_06d356715a8ff3b6 │   0.7710516467065869 │  0.21812624750499002 │
│ client_0797ff3a1fc9a6a5 │  0.08628260151865302 │                 NULL │
│ client_08a6a72ff48e62c0 │   0.3382862858311651 │                  0.0 │
│ client_08d2847f24cf89c1 │  0.13893047911945614 │ 0.036598549309527134 │
│ client_0b245132bb722950 │   0.5157936077499953 │ 0.031531110321482675 │
│ client_0e1acc6cd57b0eba │ 0.052492167473654226 │ 0.016143678740793057 │
│ client_0fa64a184f18a4a0 │   0.6662704856943366 │  0.08433555351581194 │
│ client_157ffe4d4a595515 │   0.4634365592005767 │  0.04942994547715762 │
│ client_19b89ee4fe3db6da │           

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.